# Canonical Correlation Forest and MATLAB conformance

Fit the native sklearn-compatible CCF, validate its deterministic MATLAB-source fixture, and construct only its supported original co-leaf relation. CCF-Bag OOB/RF-GAP remain gated.


All examples use local synthetic data, fixed seeds, and a reduced smoke
configuration by default. `original`, `kerf`, `path`, `random_height`, and the
separately named isolation `collision` relation are symmetric affinities.
`rfgap`, `honest_gap`, and conditional weights are directed query-to-training
weights; downstream visualization, clustering, or attribution must state any
symmetrization. Prediction identity is claimed only for unsymmetrized `rfgap`
or `honest_gap` under their declared bootstrap/honesty contracts.

In [1]:
SMOKE = True
RANDOM_STATE = 0
N_SAMPLES = 80 if SMOKE else 800
N_ESTIMATORS = 20 if SMOKE else 300

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from forestgap import CanonicalCorrelationForestClassifier, Proximity
from forestgap.adapters import (
    UnsupportedRelationError, load_ccf_oracle_fixture,
    method_capabilities, validate_ccf_oracle,
)

X, y = make_classification(
    n_samples=N_SAMPLES, n_features=6, n_informative=4,
    n_redundant=0, random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, stratify=y, random_state=RANDOM_STATE,
)
forest = CanonicalCorrelationForestClassifier(
    n_estimators=N_ESTIMATORS, max_depth=8, random_state=RANDOM_STATE,
).fit(X_train, y_train)
proximity = Proximity(forest, relation="original").attach(X_train, y_train)
P_test = proximity.prox_extend(X_test)
{"accuracy": forest.score(X_test, y_test), "original_relation_shape": P_test.shape}

{'accuracy': 0.9, 'original_relation_shape': (20, 60)}

In [3]:
fixture = load_ccf_oracle_fixture()
oracle = validate_ccf_oracle(
    CanonicalCorrelationForestClassifier(**fixture["estimator_parameters"]),
    np.asarray(fixture["X_train"]),
    np.asarray(fixture["classification"]["y_train"]),
    np.asarray(fixture["X_test"]),
    np.asarray(fixture["classification"]["expected_predictions"]),
)
capability = method_capabilities("canonical_correlation_forest")
gates = {}
for relation in ("oob", "rfgap"):
    try:
        Proximity(forest, relation=relation).attach(X_train, y_train)
    except UnsupportedRelationError as error:
        gates[relation] = {"status": "gated", "reason": error.reason}
{
    "oracle_source_commit": fixture["source"]["commit"],
    "oracle_passed": oracle.passed,
    "oracle_max_absolute_error": oracle.max_absolute_error,
    "valid_relations": capability.valid_relations,
    "gated_relations": gates,
}

{'oracle_source_commit': 'f4b7bf45958f089d5c75441b35a95d62572e4559',
 'oracle_passed': True,
 'oracle_max_absolute_error': 0.0,
 'valid_relations': ('original',),
 'gated_relations': {'oob': {'status': 'gated',
   'reason': 'The native CCF supports original co-leaf proximity; CCF-Bag OOB/RF-GAP semantics remain explicitly gated.'},
  'rfgap': {'status': 'gated',
   'reason': 'The native CCF supports original co-leaf proximity; CCF-Bag OOB/RF-GAP semantics remain explicitly gated.'}}}